In [2]:
import pandas as pd

df = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Growth and Inflation 3Q2024.xlsx"
                   , sheet_name=1
                   , header=1)
df.head()

,Region,China,Hong Kong SAR (China),India,Indonesia,South Korea,Malaysia,Mongolia,Philippines,Singapore,...,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26
0,Unit,Prev Year=100,%,INR mn,%,%,MYR mn,MNT mn,PHP mn,%,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Last Update Time,2024-10-18 00:00:00,2024-10-31 00:00:00,2024-11-29 00:00:00,2024-11-05 00:00:00,2024-12-18 00:00:00,2024-11-15 00:00:00,2024-11-18 00:00:00,2024-11-07 00:00:00,2024-11-22 00:00:00,...,India,Indonesia,South Korea,Malaysia,Mongolia,Philippines,Singapore,Taiwan,Thailand,Vietnam
2,2000-03-01 00:00:00,108.7,10.7,NaN,NaN,12.6,NaN,NaN,1653296.013866,9,...,NaN,NaN,12.6,NaN,NaN,NaN,9,NaN,NaN,NaN
3,2000-06-01 00:00:00,108.9,7.4,NaN,NaN,9.2,NaN,NaN,1729789.409909,8.2,...,NaN,NaN,9.2,NaN,NaN,NaN,8.2,NaN,NaN,NaN
4,2000-09-01 00:00:00,108.9,7.2,NaN,NaN,9.3,NaN,NaN,1709412.902916,9.5,...,NaN,NaN,9.3,NaN,NaN,NaN,9.5,NaN,NaN,NaN


In [ ]:
import openpyxl

# Load the workbook
wb = openpyxl.load_workbook("example.xlsx")

# Select the worksheet
ws = wb["Sheet1"]

# Find the last row with data
last_row = ws.max_row  # Get the last row after adding data

# Locate the chart in the worksheet
for drawing in ws._charts:
    chart = drawing  # Assuming there's only one chart; if multiple, modify accordingly

    # Update the chart's data range (Assuming it's based on column A and B)
    new_data_range = f"Sheet1!$A$2:$A${last_row},Sheet1!$B$2:$B${last_row}"

    # Update the chart series with the new range
    chart.series[0].values = new_data_range  # Assuming it's the first series

# Save the workbook with the updated chart range
wb.save("example.xlsx")

print("Chart updated successfully!")

############################################################################################################################

# Update a formula in a specific cell (e.g., C2)
ws["C2"].value = "=SUM(A2:B2)"  # Updating the formula

############################################################################################################################

# Find the last row with data
last_row = ws.max_row  # Get the last row with data

# Update the formula in the cell (assuming it's in B6, update accordingly)
ws["B{}".format(last_row)].value = f"=SUM(A2:A{last_row})"


In [3]:
def convert_date_to_quarter(date_str, dtq=True):
    if not dtq:
        year = date_str[:4]
        quarter = date_str[5:7]
        qd_dict = {'03': '1Q', '06': '2Q', '09': '3Q', '12': '4Q'}
    else:
        year = date_str[-4:]
        quarter = date_str[0]
        qd_dict = {'1': '31/3/', '2': '30/6/', '3': '30/9/', '4': '31/12/'}
    if quarter in qd_dict.keys():
        return f"{qd_dict.get(quarter)}{year}"    
    else:
        return date_str

In [4]:
fd = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Growth and Inflation 3Q2024.xlsx"
                   , sheet_name=2, nrows=12)


col_name = list(map(convert_date_to_quarter,fd.columns))
fd.rename({fd.columns[0]: 'Region'}, axis=1, inplace=True)
fd.set_index('Region', inplace=True)
fd.loc['Total'] = fd.sum()

def fd_calc(fd: pd.DataFrame, cal = True) -> pd.DataFrame:
    sum_row = fd.sum()
    fd.loc['Total'] = sum_row
    if cal:
        fd = fd.apply(lambda x: x/fd.loc['Total'], axis=1)
    return fd


In [5]:
df2 = df.copy()
df2 = df2.drop([0,1], axis=0)
ch_df = pd.read_excel(r"C:\Users\AhmadAizudeen\OneDrive - The SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\Capital Flow Monitor\SEACEN CFM January 2025\edited files\Data\Growth and Inflation 3Q2024.xlsx"
                   , sheet_name=0, header=1)
ch_df = ch_df.drop([0,1])[['Region', 'China']].set_index('Region').rename(columns={'Region': 'Date'})
df2 = df2[df2.columns[:13]].rename(columns={'Region':'Date'}).set_index('Date')
df2['China'] = ch_df['China']
df3 = df2[['China', 'India', 'Malaysia', 'Mongolia', 'Philippines', 'Taiwan', 'Thailand']].apply(lambda x : ((x/x.shift(4)) - 1)*100)
df3 = pd.concat([df2[['Hong Kong SAR (China)', 'Indonesia', 'South Korea', 'Singapore', 'Vietnam']], df3], axis=1)
df3.reset_index(inplace=True)
df3["Date"] = pd.to_datetime(df3["Date"]) + pd.offsets.MonthEnd(0)
select_row = len(fd.columns)
df4 = df3[-select_row:]


df4['Date'] = list(df4['Date'].apply(lambda x: convert_date_to_quarter(str(x), False)))
df4 = df4.set_index('Date').transpose().reset_index().rename(columns={'index':'Region'}) #this is bullshit
df4.set_index('Region', inplace=True)
df4.rename(index={'Hong Kong SAR (China)': 'Hong Kong', 'South Korea': 'Korea', 'Taiwan': 'Taipei'}, inplace=True)
df4 = df4.reindex(fd.index[:12])


c:\USERS\AHMADAIZUDEEN\ONEDRIVE - THE SOUTH-EAST ASIAN CENTRAL BANKS (SEACEN) RESEARCH AND TRAINING\DESKTOP\NLP PROJECT\VENC\Lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)
C:\Users\AhmadAizudeen\AppData\Local\Temp\ipykernel_9924\17278457.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df4['Date'] = list(df4['Date'].apply(lambda x: convert_date_to_quarter(str(x), False)))


In [9]:
asean5 = fd_calc(fd.loc[['Indonesia', 'Malaysia', 'Philippines','Thailand', 'Vietnam']]) * fd_calc(df4.loc[['Indonesia', 'Malaysia', 'Philippines','Thailand', 'Vietnam']], cal=False)
asean5.drop('Total', inplace=True)
asean5_sum = asean5.sum()
asean5.loc['ASEAN-5'] = asean5_sum

adv_asia = fd_calc(fd.loc[['Hong Kong', 'Korea', 'Singapore','Taipei']]) * fd_calc(df4.loc[['Hong Kong', 'Korea', 'Singapore','Taipei']], cal=False)
adv_asia.drop('Total', inplace=True)
adv_asia_sum = adv_asia.sum()
adv_asia.loc['Asia Advanced Economies'] = adv_asia_sum

asia = fd.apply(lambda x: x/fd.loc['Total'], axis=1) * df4
asia_sum = asia.sum()
asia.loc['Asia'] = asia_sum

gdp_growth = pd.concat([asia.loc['Asia'], df4.loc['China'], df4.loc['India'], asean5.loc['ASEAN-5'], adv_asia.loc['Asia Advanced Economies']], axis=1).reset_index().rename(columns={'index':'Quarter'})
gdp_growth = gdp_growth.melt(id_vars="Quarter", var_name="Region", value_name="Value")
gdp_growth

,Quarter,Region,Value
0,1Q2022,Asia,4.552098
1,2Q2022,Asia,4.220563
2,3Q2022,Asia,4.797955
3,4Q2022,Asia,3.267145
4,1Q2023,Asia,4.486541
5,2Q2023,Asia,5.913564
6,3Q2023,Asia,5.195059
7,4Q2023,Asia,5.671215
8,1Q2024,Asia,5.633311
9,2Q2024,Asia,5.063675


In [11]:

gdp_growth['Quarter'].unique()

'1Q2022'